In [1]:
import duckdb, numpy, functools
con = duckdb.connect(database=':memory:')
@functools.lru_cache(128)
def q(query: str):
    return con.execute(query).df()

In [66]:
%cd analysis

[Errno 2] No such file or directory: 'analysis'
/home/isaackhor/code/cfcache/analysis


In [3]:
q('''
SET memory_limit = '32GB';
''')

,Success


In [12]:
q('''
select server, count(*) / 1000000 as n
from '../traces/cf/**/*.parquet'
group by server
order by server asc
  ''')

,server,n
0,106m105,611.934154
1,106m106,737.911445
2,243m12,585.034405
3,243m13,633.171931
4,411m264,350.792601
5,411m325,390.182372
6,472m378,305.423630
7,472m379,322.921382


In [4]:
q('''
select server, count(distinct key) / 1000000 as n
from '../traces/cf/**/*.parquet'
group by server
order by server asc
  ''')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,server,n
0,106m105,39.394027
1,106m106,45.745895
2,243m12,69.164792
3,243m13,74.905332
4,411m264,31.410863
5,411m325,32.923631
6,472m378,30.637314
7,472m379,29.307364


In [ ]:
q('''
select server, sum(size) / 1024 / 1024 / 1024 as n
from '../traces/cf/**/*.parquet'
group by server
order by server asc
  ''')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,server,n
0,106m105,114112.345038
1,106m106,144711.950847
2,243m12,146132.841774
3,243m13,160366.138752
4,411m264,206125.983156
5,411m325,307594.511235
6,472m378,97255.755088
7,472m379,103416.765967


In [71]:
display(q(''' select count(*) / 1000000 as n from '../traces/fb23/reag.parquet' '''))
display(q(''' select count(*) / 1000000 as n from '../traces/fb23/rhna.parquet' '''))
display(q(''' select count(*) / 1000000 as n from '../traces/fb23/rprn.parquet' '''))

display(q(''' select count(distinct cachekey) / 1000000 as n from '../traces/fb23/reag.parquet' '''))
display(q(''' select count(distinct cachekey) / 1000000 as n from '../traces/fb23/rhna.parquet' '''))
display(q(''' select count(distinct cachekey) / 1000000 as n from '../traces/fb23/rprn.parquet' '''))

display(q(''' select sum(responseSize) / 1000000000 as n from '../traces/fb23/reag.parquet' '''))
display(q(''' select sum(responseSize) / 1000000000 as n from '../traces/fb23/rhna.parquet' '''))
display(q(''' select sum(responseSize) / 1000000000 as n from '../traces/fb23/rprn.parquet' '''))

,n
0,50.114842


,n
0,102.880355


,n
0,96.069551


,n
0,14.706047


,n
0,36.942643


,n
0,34.573903


,n
0,22689.461403


,n
0,45033.186047


,n
0,48773.110016


In [74]:
display(q(''' select count(*) / 1000000 as n from '../traces/wm/t-all.parquet' '''))
display(q(''' select count(*) / 1000000 as n from '../traces/wm/u-all.parquet' '''))

display(q(''' select count(distinct key) / 1000000 as n from '../traces/wm/t-all.parquet' '''))
display(q(''' select count(distinct key) / 1000000 as n from '../traces/wm/u-all.parquet' '''))

display(q(''' select sum(size) / 1000000000 as n from '../traces/wm/t-all.parquet' '''))
display(q(''' select sum(size) / 1000000000 as n from '../traces/wm/u-all.parquet' '''))

,n
0,197.819321


,n
0,531.222846


,n
0,17.895056


,n
0,24.480779


,n
0,6699.838311


,n
0,22034.705864


## cloudflare

In [10]:
q("""
select timestamp, key, zone, size, expiry_time, stale_time, method,
    regexp_extract(filename, '.*/([^/]+)\\.csv\\.zst', 1) as server,
    regexp_extract(mime, '^([^;]+)') as mime
from read_csv(
    '../traces/cf/csv/106m105.csv.zst', 
    delim=',',
    header=true,
    filename=true,
    strict_mode=false
    )
limit 100
""")

,timestamp,key,zone,size,expiry_time,stale_time,method,server,mime
0,1745712000,28e3c4805dc376c3,a9deca06e76f5a50,161189,300,0,0,106m105,text/html
1,1745712000,5bb1c025b485e25e,e579e303a6479bc2,242,14400,0,0,106m105,application/json
2,1745712000,5bb1c025b485e25e,e579e303a6479bc2,242,14400,0,0,106m105,application/json
3,1745712000,5bb1c025b485e25e,e579e303a6479bc2,242,14400,0,0,106m105,application/json
4,1745712000,7bc224a22b6e3b80,e579e303a6479bc2,242,14400,0,0,106m105,application/json
...,...,...,...,...,...,...,...,...,...
95,1745712000,b34593c70572bf99,4fd28d6549873177,786,604800,0,0,106m105,video/mp4
96,1745712000,96579941bee19110,89819adc3ac439b7,999740,14400,0,0,106m105,application/octet-stream
97,1745712000,2f058f9a34b47591,81457b421349d4ae,61547,604800,0,0,106m105,image/jpeg
98,1745712000,6c5cd75ebfd9ee13,764e53ff2058dd80,100104,31557600,0,0,106m105,image/webp


In [26]:
df = q("""
select count(*) as n, trim(regexp_extract(lower(mime), '^([^;]+)', 1)) as mime
from read_csv(
    '../traces/cf/csv/106m105.csv.zst', 
    delim=',',
    quote='',
    header=true,
    filename=true,
    strict_mode=false,
    parallel=false
    )
group by mime
order by n desc
limit 128
""")

df.head()

,n,mime
0,114488227,video/mp4
1,98336262,image/jpeg
2,76836627,image/webp
3,53576279,application/json
4,30038089,image/png


In [27]:
## export 

for m in ['106m105', '106m106', '243m12', '243m13', '411m264', '411m325', '472m378', '472m379']:
    con.execute(f"""
    copy (
        select
            timestamp, key, zone, size, expiry_time, stale_time, method, 
                trim(regexp_extract(lower(mime), '^([^;]+)', 1)) as mime,
                regexp_extract(filename, '.*/([^/]+)\\.csv\\.zst', 1) as server
        from read_csv(
            '../traces/cf/csv/{m}.csv.zst',
            delim=',',
            quote='', 
            comment='',
            header=true,
            strict_mode=false,
            filename=true,
            union_by_name=true,
            parallel=false
        )
    )
    to '../traces/cf/parquet/' (
        format parquet,
        partition_by ('server'),
        overwrite_or_ignore,
        compression zstd
    )
    """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## facebook

In [11]:
con.execute(f"""
select * from read_csv('../traces/fb23/reag.csv.zst', strict_mode=false)
limit 100
""").df()

,timestamp,cacheKey,OpType,objectSize,responseSize,responseHeaderSize,rangeStart,rangeEnd,TTL,SamplingRate,cache_hit,item_value,RequestHandler,cdn_content_type_id,vip_type
0,1678863509425,0067813353300000000000000000000000000000000000...,1,-1,246307,547,-1,-1,1209600,0.0001,0,1,1000,9,2000
1,1678863511382,0067813353400000000000000000000000000000000000...,1,-1,2736128,0,-1,-1,1209600,0.0001,0,1,1000,2,2000
2,1678863569821,0067813353500000000000000000000000000000000000...,1,25568,26063,450,-1,-1,1209600,0.0001,0,1,1000,15,2000
3,1678863576365,0067813353600000000000000000000000000000000000...,1,20914,23493,2561,-1,-1,1209600,0.0001,0,0,1001,6,2001
4,1678863577913,0067813353700000000000000000000000000000000000...,1,10243,11691,1448,-1,-1,900,0.0001,0,1,1002,28,2002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1678863586794,0067813359500000000000000000000000000000000000...,1,155124,155664,450,0,8388607,1209600,0.0001,1,1,1000,2,2000
96,1678863586807,0067813359500000000000000000000000000000000000...,1,155124,216,207,8388608,16777215,-1,0.0001,1,1,1000,2,2000
97,1678863587000,0067813354200000000000000000000000000000000000,1,5776,6439,663,-1,-1,1209600,0.0001,1,1,1003,3,2000
98,1678863587033,0067813360100000000000000000000000000000000000...,1,896,1637,741,-1,-1,1209600,0.0001,1,1,1001,1,2000


In [3]:
con.execute(f"""
select * from read_csv('../traces/fb23/rhna.csv.zst', strict_mode=false)
limit 100
""").df()

,timestamp,cacheKey,OpType,objectSize,responseSize,responseHeaderSize,rangeStart,rangeEnd,TTL,SamplingRate,cache_hit,item_value,RequestHandler,cdn_content_type_id,vip_type
0,1678863573318,0344441004800000000000000000000000000000000000...,1,383498,385511,1797,-1,-1,1209600,0.0004,0,0,1000,24,2000
1,1678863577546,0344441004900000000000000000000000000000000000...,1,8892372,198590,1766,1835008,2031615,1209600,0.0004,0,0,1001,16,2000
2,1678863577611,0344441005000000000000000000000000000000000000...,1,19789,22552,2745,-1,-1,1209600,0.0004,0,0,1002,6,2000
3,1678863577952,0344441005100000000000000000000000000000000000...,1,11129278,11144323,420,-1,-1,1209600,0.0004,0,1,1003,7,2001
4,1678863578111,0344441005200000000000000000000000000000000000...,1,110557540,1182748,2290,82116608,83296255,539240,0.0004,0,0,1002,2,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1678863582634,0344441013300000000000000000000000000000000000...,1,22572,25189,2599,-1,-1,1209600,0.0004,0,0,1002,9,2000
96,1678863582646,0344441013400000000000000000000000000000000000...,1,255171,255977,518,-1,-1,1209600,0.0004,0,1,1003,7,2001
97,1678863582746,0344441013500000000000000000000000000000000000...,1,42253,44935,2655,-1,-1,129600,0.0004,0,0,1001,15,2000
98,1678863582752,0344441013600000000000000000000000000000000000...,1,54758,57539,2745,-1,-1,1209600,0.0004,0,0,1002,1,2000


In [31]:
for m in ['reag', 'rhna', 'rprn']:
    display(q(f'''
    copy (select * from read_csv('../traces/fb23/{m}.csv.zst', strict_mode=false))
    to '../traces/fb23/{m}.parquet' ( format parquet, overwrite_or_ignore, compression zstd)
    '''))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,50114842


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,102880355


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,96069551


In [14]:
con.execute(f"""
select timestamp, cacheKey, OpType, objectSize, responseSize, TTL, cache_hit from read_csv('../traces/fb23/reag.csv.zst', strict_mode=false)
limit 10
""").df()

,timestamp,cacheKey,OpType,objectSize,responseSize,TTL,cache_hit
0,1678863509425,0067813353300000000000000000000000000000000000...,1,-1,246307,1209600,0
1,1678863511382,0067813353400000000000000000000000000000000000...,1,-1,2736128,1209600,0
2,1678863569821,0067813353500000000000000000000000000000000000...,1,25568,26063,1209600,0
3,1678863576365,0067813353600000000000000000000000000000000000...,1,20914,23493,1209600,0
4,1678863577913,0067813353700000000000000000000000000000000000...,1,10243,11691,900,0
5,1678863578196,0067813353800000000000000000000000000000000000...,1,52428800,4199451,1209600,0
6,1678863578513,0067813353900000000000000000000000000000000000...,1,10115,11562,900,0
7,1678863578513,0067813354000000000000000000000000000000000000...,1,11748,14286,1209600,0
8,1678863579265,0067813354100000000000000000000000000000000000...,1,115221891,115223787,1209600,0
9,1678863579339,0067813354200000000000000000000000000000000000,1,5776,6439,1209600,1


## wikimedia

In [20]:
con.execute('''
select * from read_csv('../traces/wm/cache-t-01.gz', strict_mode=false)
limit 10
''').df()

,relative_unix,hashed_host_path_query,response_size,time_firstbyte
0,86400,-719785930,12319,0.511017
1,86400,566297063,34510,0.000197
2,86400,1252933015,47354,0.000194
3,86400,707785349,11813,0.000200
4,86400,1284302613,31883,0.000189
5,86400,1184865752,19944,0.000179
6,86400,36406143,24810,0.000187
7,86400,301688409,25326,0.000214
8,86400,209252430,90247,0.000283
9,86400,-1232876025,35868,0.000195


In [ ]:
con.execute('''
copy (
    select 
        relative_unix as timestamp,
        hashed_host_path_query as key,
        response_size as size
    from read_csv('../traces/wm/cache-t-*.gz', strict_mode=false))
to '../traces/wm/t-all.csv.zst' (format csv, compression zstd, preserve_order)
''').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,197819321


In [3]:
q('''
copy (
    select 
        relative_unix as timestamp,
        hashed_host_path_query as key,
        response_size as size
    from read_csv('../traces/wm/cache-t-*.gz', strict_mode=false))
to '../traces/wm/t-all.parquet' (format parquet, compression zstd)
''')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,197819321


In [23]:
con.execute('''
select * from read_csv('../traces/wm/cache-u-01.gz', strict_mode=false)
limit 5
''').df()

,relative_unix,hashed_path_query,image_type,response_size,time_firstbyte
0,86400,-1504246911,png,705,0.000163
1,86400,-831376498,svg+xml,177,0.000665
2,86400,-1588896002,jpeg,24407,0.000164
3,86400,-911502403,jpeg,170820,0.279900
4,86400,-96995004,png,1287,0.000226


In [24]:
con.execute('''
copy (
    select 
        relative_unix as timestamp,
        hashed_path_query as key,
        response_size as size
    from read_csv('../traces/wm/cache-u-*.gz', strict_mode=false))
to '../traces/wm/u-all.csv.zst' (format csv, compression zstd)
''').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,2656114237


In [6]:
con.execute('''
copy (
    select timestamp, key, size
    from read_csv('../traces/wm/u-all.csv.zst', strict_mode=false)
    limit 531222846
) to '../traces/wm/u-all.parquet' (format parquet, compression zstd)
''').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,531222846


In [27]:
con.execute("select * from duckdb_settings() where name = 'preserve_insertion_order'").df()

,name,value,description,input_type,scope,aliases
0,preserve_insertion_order,true,Whether or not to preserve insertion order. If...,BOOLEAN,GLOBAL,[]


# min/max timestamps

In [5]:
q("""
select min(timestamp), max(timestamp), max(timestamp) - min(timestamp)
from '../traces/cf/**/*.parquet'
""")

,"min(""timestamp"")","max(""timestamp"")","(max(""timestamp"") - min(""timestamp""))"
0,1745712000,1748476799,2764799


In [ ]:
q("""
select min(timestamp), max(timestamp), max(timestamp) - min(timestamp)
from '../traces/fb23/**/*.parquet'
""")

,"min(""timestamp"")","max(""timestamp"")","(max(""timestamp"") - min(""timestamp""))"
0,1678863469090,1679554795151,691326061


In [8]:
q("""
select min(timestamp), max(timestamp), max(timestamp) - min(timestamp)
from '../traces/wm/**/*.parquet'
""")

,"min(""timestamp"")","max(""timestamp"")","(max(""timestamp"") - min(""timestamp""))"
0,86400,1814399,1727999
